In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import tensorflow as tf

sys.path.append(os.path.abspath("../"))

from util import dataset as u_dataset
from util import image as u_image

In [ ]:
def make_example(directory, label):
    masks = u_dataset.get_masks(label, "ball")
    image_feature = tf.train.Feature(
        bytes_list=tf.train.BytesList(
            value=[
                tf.io.serialize_tensor(
                    tf.reshape(
                        tf.constant(
                            u_dataset.load_image(
                                directory, label, image_format=u_image.ImageFormat.YUYV
                            )
                        ),
                        (480, 320, 4),
                    )
                ).numpy(),
            ]
        )
    )
    camera_feature = tf.train.Feature(
        bytes_list=tf.train.BytesList(
            value=[
                tf.io.serialize_tensor(
                    tf.constant(u_dataset.camera_from_label(label), dtype=tf.float32)
                ).numpy(),
            ]
        )
    )
    intrinsics_feature = tf.train.Feature(
        bytes_list=tf.train.BytesList(
            value=[
                tf.io.serialize_tensor(
                    tf.constant(u_dataset.intrinsics_from_label(label), dtype=tf.float32)
                ).numpy(),
            ]
        )
    )
    objectness_feature = tf.train.Feature(
        bytes_list=tf.train.BytesList(
            value=[
                tf.io.serialize_tensor(
                    tf.reshape(tf.cast(masks[1], dtype=tf.float32), (15, 20))
                ).numpy(),
            ]
        )
    )
    offset_feature = tf.train.Feature(
        bytes_list=tf.train.BytesList(
            value=[
                tf.io.serialize_tensor(tf.reshape(masks[0], (15, 20, 2))).numpy(),
            ]
        )
    )
    loss_mask_feature = tf.train.Feature(
        bytes_list=tf.train.BytesList(
            value=[
                tf.io.serialize_tensor(
                    tf.reshape(tf.cast(masks[2], dtype=tf.float32), (15, 20))
                ).numpy(),
            ]
        )
    )

    # Create a Features dictionary
    features = tf.train.Features(
        feature={
            "image": image_feature,
            "camera": camera_feature,
            "intrinsics": intrinsics_feature,
            "objectness": objectness_feature,
            "offsets": offset_feature,
            "loss_mask": loss_mask_feature,
        }
    )

    example = tf.train.Example(features=features)
    # print(example)

    return example

#### Write the TFRecords file


In [ ]:
def get_dataset(directory):
    # Load the dataset
    # TODO: must be divisible by 32
    labels = u_dataset.load_labels(directory)
    # labels = u_dataset.load_labels(directory)[:768]
    record_file = "../../data.tfrecords"

    with tf.io.TFRecordWriter(record_file) as writer:
        for label in labels:
            example = make_example(directory, label)
            writer.write(example.SerializeToString())

In [ ]:
example = get_dataset(
    "../../data/groundtruth/GORE23_Gott_Gott_CompetitionWalk_Default__HTWK-Robots_1stHalf_1"
)

#### Read the TFRecords file


In [ ]:
raw_dataset = tf.data.TFRecordDataset("../../data.tfrecords")

feature_description = {
    "image": tf.io.FixedLenFeature([], tf.string),
    "camera": tf.io.FixedLenFeature([], tf.string),
    "intrinsics": tf.io.FixedLenFeature([], tf.string),
    "objectness": tf.io.FixedLenFeature([], tf.string),
    "offsets": tf.io.FixedLenFeature([], tf.string),
    "loss_mask": tf.io.FixedLenFeature([], tf.string),
}


def _parse_tensor(serialized_tensor):
    return {
        "image": tf.io.parse_tensor(serialized_tensor["image"], out_type=tf.uint8),
        "camera": tf.io.parse_tensor(serialized_tensor["camera"], out_type=tf.float32),
        "intrinsics": tf.io.parse_tensor(serialized_tensor["intrinsics"], out_type=tf.float32),
        "ball": {
            "objectness": tf.io.parse_tensor(serialized_tensor["objectness"], out_type=tf.float32),
            "offsets": tf.io.parse_tensor(serialized_tensor["offsets"], out_type=tf.float32),
            "loss_mask": tf.io.parse_tensor(serialized_tensor["loss_mask"], out_type=tf.float32),
        },
    }


def _parse_function(example_proto):
    # Parse the input tf.train.Example proto using the dictionary above.
    return _parse_tensor(tf.io.parse_single_example(example_proto, feature_description))


# parsed_dataset = tf.io.parse_example(raw_dataset, feature_description)
parsed_dataset = raw_dataset.map(_parse_function)

# print(parsed_dataset)

parsed_dataset.shuffle(32, seed=42)
parsed_dataset.batch(32)

for x in parsed_dataset:
    t = x

_, ax = plt.subplots()
ax.imshow(t["image"][..., 0] / 255)
plt.show()

print(t["ball"])
